# 01 · Fetch Data

抓兩種資料：
1. 台南市村里界圖（GeoJSON / SHP）
2. 台南市立圖書館清單，地址透過 **TGOS Lite** geocoding 取得經緯度（cache 在 `data/cache/geocode.csv`）

成功後輸出：
- `data/raw/tainan_villages.geojson`
- `data/raw/tainan_libraries.csv`

In [ ]:
import json
import sys
from pathlib import Path

import geopandas as gpd
import pandas as pd
import requests

# 讓 notebook 找得到 lib/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

RAW_DIR = ROOT / "data" / "raw"
FALLBACK_DIR = ROOT / "data" / "fallback"
RAW_DIR.mkdir(parents=True, exist_ok=True)

VILLAGES_OUT = RAW_DIR / "tainan_villages.geojson"
LIBRARIES_OUT = RAW_DIR / "tainan_libraries.csv"

print(f"ROOT = {ROOT}")
print(f"RAW_DIR = {RAW_DIR}")

## 1. 村里界圖

來源：政府資料開放平台「村里界圖」（內政部國土測繪中心）

由於該資料集 URL 經常變動且檔案較大（~50MB），預設**手動下載**：

1. 開 https://data.gov.tw/ 搜尋「村里界圖」（或上 https://segis.moi.gov.tw/ ）
2. 下載 WGS84 經緯度版本（SHP 或 GeoJSON 皆可）
3. 解壓後將檔案放到 `data/raw/`，下一個 cell 會自動讀取並過濾台南市

In [ ]:
# 自動偵測 data/raw/ 內任何 .shp 或 .geojson（除了我們自己輸出的）
candidates = [
    p for p in RAW_DIR.glob("*")
    if p.suffix.lower() in (".shp", ".geojson", ".gpkg")
    and p.name != VILLAGES_OUT.name
]

if not candidates:
    raise FileNotFoundError(
        f"找不到村里界圖。請依上一個 cell 說明手動下載到 {RAW_DIR}"
    )

src = candidates[0]
print(f"讀取 {src.name}...")
gdf = gpd.read_file(src)
print(f"全國共 {len(gdf)} 個里")
print(f"欄位：{list(gdf.columns)}")
gdf.head(2)

In [ ]:
# 內政部資料的欄位名稱可能是 COUNTYCODE / COUNTY_ID / COUNTY 之一；遇到都嘗試
county_col_candidates = ["COUNTYCODE", "COUNTY_ID", "COUNTY"]
county_col = next((c for c in county_col_candidates if c in gdf.columns), None)
if county_col is None:
    raise KeyError(
        f"找不到縣市代碼欄位（試過 {county_col_candidates}）；"
        f"現有欄位：{list(gdf.columns)}"
    )

# 台南：代碼以 67 開頭或名稱含「臺南」/「台南」
if gdf[county_col].dtype == object:
    mask = gdf[county_col].astype(str).str.contains("臺南|台南", na=False) | \
           gdf[county_col].astype(str).str.startswith("67")
else:
    mask = gdf[county_col].astype(str).str.startswith("67")

tainan = gdf[mask].copy()
print(f"台南市共 {len(tainan)} 個里")

# 確保 CRS 是 WGS84
if tainan.crs is None or tainan.crs.to_epsg() != 4326:
    tainan = tainan.to_crs(epsg=4326)

# 給每個里一個穩定的 village_id；優先使用 VILLCODE
id_col = next(
    (c for c in ["VILLCODE", "VILLAGE_ID", "VILLAGE_CODE"] if c in tainan.columns),
    None,
)
if id_col is None:
    tainan["village_id"] = [f"tn_{i:04d}" for i in range(len(tainan))]
else:
    tainan["village_id"] = tainan[id_col].astype(str)

# 統一里名 / 區名欄位
name_col = next((c for c in ["VILLNAME", "VILLAGE", "NAME"] if c in tainan.columns), None)
district_col = next((c for c in ["TOWNNAME", "TOWN"] if c in tainan.columns), None)

tainan["village_name"] = tainan[name_col] if name_col else ""
tainan["district"] = tainan[district_col] if district_col else ""

# 只保留必要欄位 + 幾何
out = tainan[["village_id", "village_name", "district", "geometry"]]
out.to_file(VILLAGES_OUT, driver="GeoJSON")
print(f"✅ Saved {len(out)} villages to {VILLAGES_OUT}")

## 2. 圖書館清單

來源：`data/fallback/libraries_hardcoded.json`（地址只）。

對每個地址呼叫 **TGOS Lite 地址定位 API**（公開憑證，免註冊）取得 WGS84 經緯度，結果寫入 `data/cache/geocode.csv` cache。

TGOS 找不到的地址 fallback 到該區質心（從 `tainan_villages.geojson` 推算），並在 `geocode_source` 欄位標記為 `district_centroid_fallback`。

若 JSON 內某筆已有手動 override 的 `lat`/`lon` 欄位，會直接採用而不查 TGOS。

In [ ]:
import json

from lib.geocoder import TGOSGeocoder

CACHE_DIR = ROOT / "data" / "cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
GEOCODE_CACHE = CACHE_DIR / "geocode.csv"

# 1. Load address list
with (FALLBACK_DIR / "libraries_hardcoded.json").open(encoding="utf-8") as f:
    raw = json.load(f)["libraries"]
print(f"Loaded {len(raw)} library entries from JSON")

# 2. Build a district-centroid fallback table from the village geojson
villages = gpd.read_file(VILLAGES_OUT)
district_centroids = {}
for d, sub in villages.groupby("district"):
    c = sub.geometry.union_all().centroid
    district_centroids[d] = (c.y, c.x)  # (lat, lon)

# 3. Geocode each library via TGOS (cached on disk)
geocoder = TGOSGeocoder(cache_path=GEOCODE_CACHE)

records = []
n_override = n_tgos = n_fallback = 0
for entry in raw:
    name, district, address = entry["name"], entry["district"], entry["address"]
    if "lat" in entry and "lon" in entry:
        lat, lon, src = entry["lat"], entry["lon"], "manual_override"
        n_override += 1
    else:
        result = geocoder.geocode(address)
        if result is not None:
            lat, lon = result
            src = "tgos"
            n_tgos += 1
        else:
            lat, lon = district_centroids[district]
            src = "district_centroid_fallback"
            n_fallback += 1
    records.append({
        "name": name, "district": district, "address": address,
        "lat": lat, "lon": lon, "geocode_source": src,
    })

libs = pd.DataFrame(records)
print(f"Resolved: {n_tgos} via TGOS, {n_override} manual override, {n_fallback} fallback to district centroid")
if n_fallback:
    print(f"Fallback entries:")
    for r in records:
        if r["geocode_source"] == "district_centroid_fallback":
            print(f"  - {r['name']} ({r['address']})")

# 4. Write the canonical libraries CSV
libs[["name", "district", "address", "lat", "lon", "geocode_source"]].to_csv(
    LIBRARIES_OUT, index=False, encoding="utf-8-sig"
)
print(f"✅ Saved {len(libs)} libraries to {LIBRARIES_OUT}")
libs.head()

In [ ]:
import matplotlib.pyplot as plt

villages = gpd.read_file(VILLAGES_OUT)
libs_gdf = gpd.GeoDataFrame(
    libs,
    geometry=gpd.points_from_xy(libs.lon, libs.lat),
    crs="EPSG:4326",
)

fig, ax = plt.subplots(figsize=(8, 9))
villages.boundary.plot(ax=ax, linewidth=0.2, color="gray")
libs_gdf.plot(ax=ax, color="red", markersize=20)
ax.set_title(f"Tainan: {len(villages)} villages + {len(libs_gdf)} libraries")
ax.set_aspect("equal")
plt.show()